### Chargement du DataFrame

In [103]:
import pandas as pd
ENTRAINEMENT = "regex"

if ENTRAINEMENT == "regex":
    df = pd.read_csv("../data/processed/dataset_annotated_regex.csv")
else:
    df = pd.read_csv("../data/processed/dataset_avis.csv")

labels = ["qualité produit", "service livraison", "service client"]

df_annotated = df[df[labels].sum(axis=1) > 0]
df.shape

(11449, 16)

### Distribution des classes

In [104]:
class_distribution = df_annotated[labels].sum().to_frame(name="nb_commentaires")
class_distribution["pourcentage"] = (
    class_distribution["nb_commentaires"] / len(df_annotated) * 100
)

class_distribution

,nb_commentaires,pourcentage
qualité produit,5340,46.641628
service livraison,6971,60.887414
service client,4569,39.907415


### Affichage d'avis au hasard

In [17]:
samples = []

pd.set_option("display.max_colwidth", None)

df_annotated = df_annotated.drop(columns=["client"], errors="ignore")

for label in labels:
    sample_df = df_annotated[df_annotated[label] == 1].sample(
        n=3,
        random_state=None
    )
    
    sample_df = sample_df.copy()
    sample_df["label_cible"] = label  # pour savoir pourquoi il est sélectionné
    
    samples.append(sample_df)

result = pd.concat(samples)

display_cols = result.rename(columns={
    "qualité produit": "produit",
    "service livraison": "livraison",
    "service client": "client"
})

display_cols = display_cols[[
    "label_cible",
    "produit",
    "livraison",
    "client",
    "clean_comment"
]]

display(display_cols)

# for _, row in display_cols.iterrows():
#     print("─" * 100)
#     print(f"Label cible : {row['label_cible']}")
#     print(f"Produit    : {row['produit']}")
#     print(f"Livraison  : {row['livraison']}")
#     print(f"Client    : {row['client']}")
#     print("\nCommentaire :")
#     print(row["clean_comment"])

,label_cible,produit,livraison,client,clean_comment
11309,qualité produit,1,0,0,au bout de même pas une semaine un écouteur ne fonctionne plus .
3677,qualité produit,1,0,1,magnifique robe . tissu de très bonne qualité . la robe tombe bien . prendre sa taille normale . a recommander ++++
4384,qualité produit,1,1,0,livraison conforme à ma commande . pas déçue . merci
54,service livraison,1,1,0,"a fuir , j'ai acheté un matelas a 400balles soit disant à moins de 70 % que chez le fabricant et livraison payante 20 eur.aparament ils savaient pas que plus le montant est énorme plus la livraison doit être gratuite ou minime.je me suis dit après tout c'est une bonne affaire je leurs ai fait confiance mais après avoir acheté j'ai consulté le site du fabricant et à ma grande surprise , il est au même prix que showroom et ils proposent la livraison gratuite et retour dans 100jours , alors que avec eux , tu n'as que 14jiurs pour le retour.le comble , il disaient que la livraison est sur rendez vous à partir du 19juin et que le livreur ne livrera que si le rendez-vous est confirmé.un jour , le 08 juin , je suis à 40 km loin de la maison , un livreur hyper pressé m'appelle et veut a tout prix livrer la commande , il ne veut ni revenir un autre jour ni la déposer en point relais . j'ai dû improviser et embêter d'autres personnes pour pouvoir récupérer la commande.qu'attendre d'un site qui vous dupe en mettant des prix gonflés , qui vous livrent une grosse commande sans vous tenir au courant et qui ne mettent pas à jour les informations de suivi de commandefranchement je désinstalle l'appli et je me désabonne de leurs mailing liste"
986,service livraison,0,1,0,"bonjourmon probleme a moi , il y a 4 jours la vente cop.copine sort et je remarque un pantalon qui me plait a 49e et une chemise a 35e.je mets les articles en favoris et pendant 3 jours je regarde mes favoris . jeudi matin wahou moins 10e avec le code wakeup.du coup je fonce je passe commande , meme ma banque m envoie le code de securité , je le met et la commande refusée . je me dit pas grave ca arrive je me suis peut-etre trompé dans le code je recommance et la je vois que je n ai plus droit au code wakeup car il est valable qu une seule fois . j appelle showroomprivé je leur explique mon cas la personne me dit on va vous alloué un autre code . l aprés_midi je peux enfin passer commande avec mon bon de réduction de 10e et la que vois je que le pantalon a augmenté de 10e . ils m expliquent que quelqu un a fait une erreur dans le prix et je leur explique que vu que j ai vu le prix a 49e on doit me le laisser a ce prix la .on me reponds faites commande et revenez vers nous .si c est pour avoir un bon d achat de 10e sur une commande ca ne m interresse pas.je leur ais donné mon numéro de portable et j attends qu on me rapelle mais je sais qu ils ne le feront pas et je tiens bon je ne passerais pas de commande et je pense que comme beaucoup d entre vous je supprimer l aplication . geneviève"
14397,service livraison,0,1,0,plus que decue par ce site qui au depart avait un sevice client et apres vente correct et qui maintenant est deplorablede gros pb de livraisons de commandes livrees partiellement avec des aticles non livres qui reapparaissent dans les ventes en cours et ne parlons pas des remboursements pour lesquels il faut sans arret se battreinadmissible ....
2176,service client,0,1,1,"voleur catastrophique cette société surtout ne commandez jamais rien de cher car il ne respecte pas les délais de remboursements ! ! ! j'ai passé une commande de vaisselle guy degrenne d'une valeur de 271euros en recevant la vaisselle j'ai eu un choc , la vaisselle était uniquement constitué de rebuts.j ai donc tout retourné et depuis impossible d'être remboursé , il disent avoir reçu un premier colis , puis les autres soit disant apres `` bizarre '' il a fallut que je prouve que j'avais tout envoyé sinon c'était dans le baba ( gardez bien toutes les preuves d'envoies ) puis depuis ils changent sans cesse l

### Création des variables X_test/y

In [105]:
X_text = df_annotated["clean_comment"].values
y = df_annotated[[
    "qualité produit",
    "service livraison",
    "service client"
]].values

### Encoding de X_text

In [106]:
from sentence_transformers import SentenceTransformer

model_emb = SentenceTransformer(
    "dangvantuan/french-document-embedding",
    trust_remote_code=True
)

X_embeddings = model_emb.encode(
    X_text,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/358 [00:00<?, ?it/s]

### Séparation des données en train/test

In [107]:
from sklearn.model_selection import train_test_split
import numpy as np

indices = np.arange(len(df_annotated))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_embeddings,
    y,
    indices,
    test_size=0.2,
    random_state=42
)

### Chargement ou création du modèle

In [108]:
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from joblib import dump, load

if ENTRAINEMENT == "regex":
    MODEL_PATH = Path("../models/LROneVsRestClassifier_regex.joblib")
else:
    MODEL_PATH = Path("../models/LROneVsRestClassifier.joblib")

if MODEL_PATH.exists():
    print("Modèle trouvé : chargement")
    clf = load(MODEL_PATH)

else:
    print("Aucun modèle trouvé : entraînement")

    log_reg = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced"
    )

    clf = OneVsRestClassifier(log_reg)
    clf.fit(X_train, y_train)

    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    dump(clf, MODEL_PATH)

    print("Modèle entraîné et sauvegardé")

Modèle trouvé : chargement


### CrossVal

In [8]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score

scorer = make_scorer(f1_score, average="micro")

cv_scores = cross_val_score(
    clf,
    X_embeddings,
    y,
    cv=5,
    scoring=scorer,
    n_jobs=-1
)

print("F1 micro par fold :", cv_scores)
print("F1 micro moyen   :", cv_scores.mean())
print("Écart-type       :", cv_scores.std())

F1 micro par fold : [0.85271318 0.88425282 0.8725004  0.8750199  0.84805954]
F1 micro moyen   : 0.8665091685183292
Écart-type       : 0.013812187657302206


### Calcul des probas sur test ou données labelisées

In [114]:
EVAL_MODE = "labellise"

if EVAL_MODE == "test":
    print("Évaluation sur le jeu de test")

    X_eval = X_test
    y_eval = y_test
elif EVAL_MODE == "labellise":
    print("Évaluation sur les données labelisées")

    df_eval = pd.read_csv("../data/processed/100_avis_annote.csv", sep=";")

    X_eval_text = df_eval["clean_comment"].values
    y_eval = df_eval[[
        "qualité produit",
        "service livraison",
        "service client"
    ]].values
    X_eval = model_emb.encode(
        X_eval_text,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

y_proba = clf.predict_proba(X_eval)

Évaluation sur les données labelisées


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

### Affichage des résultats

In [115]:
from sklearn.metrics import classification_report

threshold = 0.5

y_pred = (y_proba >= threshold).astype(int)

print(classification_report(
    y_eval,
    y_pred,
    target_names=[
        "qualité produit",
        "service livraison",
        "service client"
    ]
))

                   precision    recall  f1-score   support

  qualité produit       0.58      0.80      0.67        41
service livraison       0.67      0.38      0.48        21
   service client       0.75      0.33      0.46        27

        micro avg       0.62      0.56      0.59        89
        macro avg       0.67      0.51      0.54        89
     weighted avg       0.65      0.56      0.56        89
      samples avg       0.47      0.44      0.44        89



c:\IA\trustpilot\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\IA\trustpilot\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\IA\trustpilot\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [116]:
import numpy as np

labels = ["qualité produit", "service livraison", "service client"]

threshold = 0.50

thresholds = {
    "qualité produit": threshold,
    "service livraison": threshold,
    "service client": threshold
}

y_pred = np.zeros_like(y_proba, dtype=int)

for i, label in enumerate(labels):
    y_pred[:, i] = (y_proba[:, i] >= thresholds[label]).astype(int)

from sklearn.metrics import confusion_matrix

for i, label in enumerate(labels):
    tn, fp, fn, tp = confusion_matrix(
        y_eval[:, i],
        y_pred[:, i]
    ).ravel()
    
    df_cm = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )
    
    print(f"\n{label}")
    display(df_cm)




qualité produit


,Prédit 0,Prédit 1
Vrai 0,35,24
Vrai 1,8,33



service livraison


,Prédit 0,Prédit 1
Vrai 0,75,4
Vrai 1,13,8



service client


,Prédit 0,Prédit 1
Vrai 0,70,3
Vrai 1,18,9


In [117]:
if EVAL_MODE == "test":
    df_eval = df_annotated.iloc[idx_test].copy()
elif EVAL_MODE == "labeled":
    df_eval = df_eval.copy()
pd.set_option("display.max_colwidth", None)
for i, label in enumerate(labels):
    df_eval[f"y_true_{label}"] = y_eval[:, i]
    df_eval[f"y_pred_{label}"] = y_pred[:, i]
    df_eval[f"proba_{label}"] = y_proba[:, i]

# Affichage de quelques avis (faux négatifs ou faux positifs)
def show_errors(df, label, n=10):
    y_true = f"y_true_{label}"
    y_pred = f"y_pred_{label}"

    proba_cols = [f"proba_{l}" for l in labels]

    fn = df[(df[y_true] == 1) & (df[y_pred] == 0)]
    fp = df[(df[y_true] == 0) & (df[y_pred] == 1)]

    print(f"\n{label.upper()} — Faux négatifs")
    display(fn[["clean_comment"] + proba_cols].head(n))

    print(f"\n{label.upper()} — Faux positifs")
    display(fp[["clean_comment"] + proba_cols].head(n))

for label in labels:
    show_errors(df_eval, label, n=10)


QUALITÉ PRODUIT — Faux négatifs


,clean_comment,proba_qualité produit,proba_service livraison,proba_service client
24,"parfait comme sur la photo repond a mes attente , par contre la pampille qui etait accroché au sac , deja perdu sans m en rendre compte c est dommage",0.446971,0.495793,0.073326
39,"bonjour , pour l'anniversaire de mon père j'ai décidé de commander des bières sur vente privé , mauvaise surprise en arrivant ( avec plus d'un mois d'attente sois disant passant ) 3 sur 6 bières sont vides ... aucune vérification n'est faite au moment de l'envoi , par conséquent je ne souhaite plus acheter chez eux . pour me dédommager ils m'ont proposé un `` bon d'achat '' 5€ ( montant des frais de port ) cela m'oblige donc à recommander chez eux . c'est donc pour moi de la achat forcée ! prenez garde avant de commander .",0.225483,0.703553,0.477347
47,j'ai commandé un sac à main noir et j'ai reçu un sac vert foncé . mais je le garde en attendant de nouvelles ventes pour en prendre un noir .,0.487132,0.251469,0.201274
58,les bouteilles de 250 ml sont vraiment petites ... je ne me suis pas rendue compte . du coup je suis un peu déçue de la marchandise . sinon le service est très bien . merci,0.430964,0.569592,0.448366
61,manque article lors de la commande qui était disponible et 1 article décousu à 1 endroit,0.259710,0.542391,0.087862
62,"j'ai commandé un collier et j'en ai reçu un autre qui n'avait rien à voir . j'ai contacté le service clientèle qui m ' a dit qu'ils ne pouvaient pas m'envoyer le bon collier vu que la campagne était terminée . c'est vraiment n'importe quoi , on commande pas des trucs pour être en suspens de ce qu'on va recevoir .",0.326770,0.371474,0.911234
87,"la logistique et le service sont toujours au rendez-vous , bien ficelés . les articles de ma premirere commande sur irl sont tres moyens ! !",0.475652,0.617113,0.210083
91,reçu bien plus tôt que prévu . très contente du produit,0.408232,0.153461,0.046610



QUALITÉ PRODUIT — Faux positifs


,clean_comment,proba_qualité produit,proba_service livraison,proba_service client
0,très bonne expérience . ras,0.642241,0.017376,0.160652
3,très satisfaite de ma commande,0.544468,0.139632,0.126380
4,tout s'est très bien passé,0.551816,0.010645,0.026722
10,boîtes de chaussures défoncées mais chaussures en très bon état quand même,0.724268,0.578280,0.023044
14,content de ma commande scott,0.537495,0.078241,0.081361
21,dommage de ne pas avoir reçu le bon article . showroom est plutôt bien réputé pour faire des erreurs comme ça . joyeux noël charline,0.574488,0.266042,0.179522
22,pas terrible ! ! ! j étais bonne cliente et maintenant je commande très peu et il trouve dle moyen de se tromper d article ! ! ! ! au revoir showroom ..........,0.549204,0.152891,0.507333
23,"hormis les 2 mois d'attente , les articles correspondent à ma commande",0.589493,0.289830,0.239166
29,satisfaite 100/100de ma commande merci,0.550662,0.072261,0.116982
33,délai respecté produit bien emballé,0.778706,0.683801,0.010494



SERVICE LIVRAISON — Faux négatifs


...

,clean_comment,proba_qualité produit,proba_service livraison,proba_service client
58,les bouteilles de 250 ml sont vraiment petites ... je ne me suis pas rendue compte . du coup je suis un peu déçue de la marchandise . sinon le service est très bien . merci,0.430964,0.569592,0.448366
60,"quand l'article ne vous convient pas , les frais de retour sont à votre charge , donc pour un manteau et un pantalon , j en ai eu pour 14 euros de frais d expédition.les autres sites similaires , les retours sont gratuits , donc fini pour moi , mon showroom privé .",0.634869,0.560305,0.255435
61,manque article lors de la commande qui était disponible et 1 article décousu à 1 endroit,0.259710,0.542391,0.087862
93,minelli paris don le site e don la boite c ' è minelli espande pourqoi ?,0.336770,0.529470,0.099839



SERVICE CLIENT — Faux négatifs


,clean_comment,proba_qualité produit,proba_service livraison,proba_service client
7,"après réflexion j ’ ai souhaité modifier ma commande et ajouter un article résultat impossible du coup frais de port payé deux fois pour la même vente ... dommage , ni économique , ni écologique .",0.391140,0.299883,0.479488
8,les chaussures reçues ne sont pas de la marque commandée ! papillio au lieu de birkenstock ! et sans être prévenue !,0.797997,0.061830,0.262568
11,bonjour j'ai reçu deux articles qui correspondent pas à ma commande la robe rouge évasée j'ai reçu carrément un autre article de la marque etam et la blouse le coloris n'est pas du tout la même que celle de l'image bien à vous,0.813976,0.129424,0.093014
12,bien que j'ai été prévenue il manquait un élément à ma commande et c'est malheureusement pour le produit en question que je passais commande .,0.294647,0.282580,0.380979
13,bonjourj'ai commandé un tableau la commande est toujours en préparation depuis 15 jours site à éviter 😡,0.268922,0.073820,0.452442
15,"commande n°205309555 d'un sac laura ashley , le sac comporte un gros coup de cuter dans le bas . commande n°205309554 des braseros , il manque les vis de montage . très déçu du sérieux de ce pure player .",0.850567,0.313183,0.173794
21,dommage de ne pas avoir reçu le bon article . showroom est plutôt bien réputé pour faire des erreurs comme ça . joyeux noël charline,0.574488,0.266042,0.179522
23,"hormis les 2 mois d'attente , les articles correspondent à ma commande",0.589493,0.289830,0.239166
31,je n'ai pas reçu l l'article commandé j'ai reçu un chemisier à carreaux marron et noir à la place d un chemisier blanc,0.382203,0.448557,0.036233
39,"bonjour , pour l'anniversaire de mon père j'ai décidé de commander des bières sur vente privé , mauvaise surprise en arrivant ( avec plus d'un mois d'attente sois disant passant ) 3 sur 6 bières sont vides ... aucune vérification n'est faite au moment de l'envoi , par conséquent je ne souhaite plus acheter chez eux . pour me dédommager ils m'ont proposé un `` bon d'achat '' 5€ ( montant des frais de port ) cela m'oblige donc à recommander chez eux . c'est donc pour moi de la achat forcée ! prenez garde avant de commander .",0.225483,0.703553,0.477347



SERVICE CLIENT — Faux positifs


,clean_comment,proba_qualité produit,proba_service livraison,proba_service client
9,un peu déçue de valider une commande et que finalement un article ne soit pas dispo .,0.316637,0.261182,0.524049
42,j ’ aime bien le service,0.241101,0.097747,0.649716
80,"service très agréable , mais je suis déçu car le haut sans boutons .",0.750340,0.037050,0.621770
